# Wave exposure pipeline

Runs the full classification pipeline using the `waves` package.

In [3]:
import geopandas as gpd
from osgeo import gdal, ogr, osr
import numpy as np
import waves
from waves.paths import CLASSIFIED, CLASSIFIED_SIEVED, RAW_VECTOR, CLEANED_VECTOR

## 1. Prepare raster data

In [4]:
waves.preprocess.bolge_model_data()

Rasterizing aoi mask...
70...80...90...100 - done in 00:07:45.
AOI mask ready: /home/kim/work/eswm-bolgemodell/niva/tmp_outline_mask.tif
Computing AOI proximity for padded clip...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:06:31.
AOI proximity ready: /home/kim/work/eswm-bolgemodell/niva/tmp_aoi_proximity.tif
Copying raster for filling...
Filling nodata (0) inside outline...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:27:49.
Coarse fill: downsampling 20× ...
Coarse fill: FillNodata at low resolution...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:13.
Coarse fill: upsampling back to original resolution...
Coarse fill: merging into raster (remaining nodata only)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:27.
Clipping filled raster to AOI + 2-pixel padding...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:10.
Filled COG saved: /home/kim/work/eswm-bolgemodell/niva/EswmRast

## 2. Inspect NiN 3 classification table

In [5]:
waves.classify.CLASSES


,class_int,trinn,navn_no,navn_en,swm_lower,swm_upper
0,1,0,minimal vannforstyrrelsesintensitet,still water,NaN,1200.0
1,2,a,svært beskyttet,very sheltered,1200.0,4000.0
2,3,b,temmelig beskyttet,moderately sheltered,4000.0,10000.0
3,4,c,litt beskyttet,slightly sheltered,10000.0,50000.0
4,5,d,svakt eksponert,weakly sheltered,50000.0,100000.0
5,6,e,litt eksponert,slightly exposed,100000.0,500000.0
6,7,f,temmelig eksponert,moderately exposed,500000.0,1000000.0
7,8,g,svært eksponert,very exposed,1000000.0,2000000.0
8,9,h,ekstremt eksponert,extremely exposed,2000000.0,4000000.0
9,10,y,disruptivt eksponert,disruptively exposed,4000000.0,NaN


## 3. Classify, sieve, and polygonize

Create a raster with ints for each of the NiN classes

In [ ]:

waves.classify.reclassify_raster(waves.paths.FILLED_CLIPPED_COG, waves.paths.CLASSIFIED_CLIPPED)

## 4. Save direct raster -> vector


In [ ]:

def create_raw_polygons(input_raster, output_gpkg, epsg_code, nodata=0):
    ds = gdal.Open(input_raster)
    band = ds.GetRasterBand(1)

    # Build mask band block-by-block to avoid loading the full raster into memory
    drv_mem = gdal.GetDriverByName("Mem")
    mask_ds = drv_mem.Create("", ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Byte)
    mask_ds.SetGeoTransform(ds.GetGeoTransform())
    mask_ds.SetProjection(ds.GetProjection())
    mask_band = mask_ds.GetRasterBand(1)

    block_x, block_y = band.GetBlockSize()
    for y0 in range(0, ds.RasterYSize, block_y):
        bh = min(block_y, ds.RasterYSize - y0)
        for x0 in range(0, ds.RasterXSize, block_x):
            bw = min(block_x, ds.RasterXSize - x0)
            block = band.ReadAsArray(x0, y0, bw, bh)
            mask_band.WriteArray((block != nodata).astype(np.uint8), x0, y0)

    # Prepare output GeoPackage
    srs = osr.SpatialReference()
    srs.ImportFromEPSG(epsg_code)

    drv_gpkg = ogr.GetDriverByName("GPKG")
    out_ds = drv_gpkg.CreateDataSource(output_gpkg)
    out_layer = out_ds.CreateLayer("polys", srs=srs, geom_type=ogr.wkbPolygon)
    out_layer.CreateField(ogr.FieldDefn("DN", ogr.OFTInteger))

    gdal.Polygonize(band, mask_band, out_layer, 0, [], callback=None)

    mask_ds = out_ds = ds = None
    print(f"Polygons saved to {output_gpkg}")


create_raw_polygons(str(waves.paths.CLASSIFIED_CLIPPED), str(waves.paths.DIRECT_VECTOR), 25833, nodata=0)


Polygons saved to /home/kim/work/eswm-bolgemodell/niva/EswmVectorDirect.gpkg


In [ ]:
gdf = gpd.read_file(waves.paths.DIRECT_VECTOR).rename(columns={"DN": "class_int"})
gdf = waves.classify.add_class_attributes(gdf)
gdf.head()


,class_int,geometry,trinn,navn_no,navn_en
0,7,"POLYGON ((953841.631 7939956.561, 953841.631 7...",f,temmelig eksponert,moderately exposed
1,6,"POLYGON ((953841.631 7939931.561, 953841.631 7...",e,litt eksponert,slightly exposed
2,7,"POLYGON ((973741.631 7939831.561, 973741.631 7...",f,temmelig eksponert,moderately exposed
3,7,"POLYGON ((953541.631 7939806.561, 953541.631 7...",f,temmelig eksponert,moderately exposed
4,6,"POLYGON ((953591.631 7939781.561, 953591.631 7...",e,litt eksponert,slightly exposed


In [ ]:
gdf.to_file(waves.paths.DIRECT_VECTOR, driver="GPKG")

## 5. Produce a map vector dataset

The below steps produces a vector data set that it easier to work with and with good coverage of tørrfall

In [ ]:
waves.classify.reclassify_raster(waves.paths.FILLED_COG, waves.paths.CLASSIFIED)
print(f"Classified raster saved: {CLASSIFIED}")
waves.classify.sieve_filter(CLASSIFIED, CLASSIFIED_SIEVED, threshold=4)
print(f"Sieved raster saved: {CLASSIFIED_SIEVED}")
waves.vectorize.vectorize_raster(CLASSIFIED_SIEVED, RAW_VECTOR)
print(f"Vectorized polygons saved: {RAW_VECTOR}")

0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:41.
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:40.
Classified raster saved: /home/kim/work/eswm-bolgemodell/niva/EswmRaster_classified.tif
Copying classified raster...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:11.
Applying sieve filter (threshold=4, connectedness=8)...
0...10...20...30...40...50...60...70...80...90...100 - done in 00:01:14.
Sieved raster saved: /home/kim/work/eswm-bolgemodell/niva/EswmRaster_classified_sieved.tif
Vectorizing /home/kim/work/eswm-bolgemodell/niva/EswmRaster_classified_sieved.tif (EPSG:32633) …
Importing raster into GRASS …


Importing raster map <raster>...
   0   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Extracting areas...
   0

Vectorizing (r.to.vect -s) …


   3   6   9  12  15  18  21  24  27  30  33  36  39  42  45  48  51  54  57  60  63  66  69  72  75  78  81  84  87  90  93  96  99 100
Writing areas...
   0   4   8  12  16  20  24  28  32  36  40  44  48  52  56  60  64  68  72  76  80  84  88  92  96 100
Building topology for vector map <vector@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    5800    5900    6000    6100    6200    6300    6400    6500    6600    6700    6800    6900    7000
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  4

Generalizing with Douglas (threshold=12.0 m) …


Copying features...
   2   5   8  11  14  17  20  23  26  29  32  35  38  41  44  47  50  53  56  59  62  65  68  71  74  77  80  83  86  89  92  95  98 100
Building topology for vector map <vector_smooth@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    5800    5900    6000    6100    6200    6300    6400    6500    6600    6700    6800    6900    7000
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  

Exporting to /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg …


         category are written only when -c flag is given.


Done → /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg
Vectorized polygons saved: /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg


## 6. Merge small polygons

Merge polygons smaller than 624 m² into their largest neighbour (`v.clean rmarea`).

In [7]:

waves.vectorize.merge_small_polygons(RAW_VECTOR, CLEANED_VECTOR, threshold=624)
print(f"Merged polygons saved: {CLEANED_VECTOR}")

Merging small polygons (threshold=624 m²) in /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_raw.gpkg …
Importing vector into GRASS …


Check if OGR layer <wave_exposure> contains polygons...
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
Creating attribute table for layer <wave_exposure>...
Column name <cat> renamed to <cat_>
Importing 207909 features (OGR layer <wave_exposure>)...
   0   2   4   6   8  10  12  14  16  18  20  22  24  26  28  30  32  34  36  38  40  42  44  46  48  50  52  54  56  58  60  62  64  66  68  70  72  74  76  78  80  82  84  86  88  90  92  94  96  98 100
-----------------------------------------------------
Registering primitives...
-----------------------------------------------------700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200
Cleaning polygons
-----------------------------------------------------
Breaking polygons...
Breaking polygons (pass 1: select bre

Merging small areas (v.clean rmarea, threshold=624.0 m²) …


--------------------------------------------------
Tool: Threshold
Remove small areas: 624
--------------------------------------------------
Copying features...
   2   5   8  11  14  17  20  23  26  29  32  35  38  41  44  47  50  53  56  59  62  65  68  71  74  77  80  83  86  89  92  95  98 100
Rebuilding parts of topology...
Building topology for vector map <merged_polygons@PERMANENT>...
Registering primitives...
Building areas...    300     400     500     600     700     800     900    1000    1100    1200    1300    1400    1500    1600    1700    1800    1900    2000    2100    2200    2300    2400    2500    2600    2700    2800    2900    3000    3100    3200    3300    3400    3500    3600    3700    3800    3900    4000    4100    4200    4300    4400    4500    4600    4700    4800    4900    5000    5100    5200    5300    5400    5500    5600    5700    5800    5900    6000    6100    6200    6300    6400    6500    6600    6700    6800    6900    7000
   0   2   4   6  

Done → /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_merged.gpkg
Merged polygons saved: /home/kim/work/eswm-bolgemodell/niva/Eswm_classified_merged.gpkg


         category are written only when -c flag is given.


## 7. Subtract land

Load a previously saved checkpoint, or run `subtract_land` to (re)compute it.

In [ ]:
waves.preprocess.subtract_land()

## 8. Add NiN class attributes

In [3]:
gdf = gpd.read_file(waves.paths.LAND_CLIPPED_VECTOR)
gdf = waves.classify.add_class_attributes(gdf)
gdf.head()


,cat,cat_,class_int,label,geometry,trinn,navn_no,navn_en
0,63736,405,6,None,"MULTIPOLYGON (((978291.631 7931119.061, 978266...",e,litt eksponert,slightly exposed
1,63750,433,5,None,"MULTIPOLYGON (((976766.631 7930881.561, 976729...",d,svakt eksponert,weakly sheltered
2,63761,440,2,None,"MULTIPOLYGON (((962416.631 7930844.061, 962391...",a,svært beskyttet,very sheltered
3,63805,514,5,None,"MULTIPOLYGON (((949979.131 7930006.561, 949970...",d,svakt eksponert,weakly sheltered
4,63816,523,5,None,"MULTIPOLYGON (((977937.505 7929817.685, 977936...",d,svakt eksponert,weakly sheltered


In [ ]:
gdf_fin = gdf.drop(columns=[c for c in gdf.columns if c in ["cat", "cat_", "label"]])

gdf_fin.to_file(waves.paths.SEA_CLIPPED_VECTOR, driver="GPKG")
